In [1]:
import os
import mlflow
import dagshub
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

os.environ['DAGSHUB_USER_TOKEN'] = '6a076dbb1debcdd8e5e02d9e0bf45faf0b5adb13'
dagshub.init(repo_owner="lchit22", repo_name="ml-assignment-house-prices", mlflow=True)

train = pd.read_csv('Data/train.csv')
test  = pd.read_csv('Data/test.csv')
test_ids = test['Id'].copy()

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

Accessing as lchit22

Initialized MLflow to track repo "lchit22/ml-assignment-house-prices"

Repository lchit22/ml-assignment-house-prices initialized!

Train shape: (1460, 81)
Test shape:  (1459, 80)


In [8]:
def clean_approach2(df):
    df = df.copy()
    none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                 'BsmtFinType2', 'MasVnrType']
    for col in none_cols:
        if col in df.columns:
            df[col] = df[col].fillna('None')
    zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
                 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                 'TotalBsmtSF', 'MasVnrArea']
    for col in zero_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )
    num_cols = df.select_dtypes(include='number').columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    cat_cols = df.select_dtypes(include='object').columns
    df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])
    return df

def engineer_features(df):
    df = df.copy()
    df['TotalSF']           = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge']          = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    df['TotalBathrooms']    = (df['FullBath'] + df['BsmtFullBath'] +
                               0.5 * df['HalfBath'] + 0.5 * df['BsmtHalfBath'])
    df['HasPool']           = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']         = (df['GarageArea'] > 0).astype(int)
    df['HasBasement']       = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace']      = (df['Fireplaces'] > 0).astype(int)
    df['TotalPorchSF']      = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                               df['3SsnPorch'] + df['ScreenPorch'])
    return df

def encode_features(df):
    df = df.copy()
    quality_map  = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    quality_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                    'HeatingQC', 'KitchenQual', 'FireplaceQu',
                    'GarageQual', 'GarageCond', 'PoolQC']
    for col in quality_cols:
        if col in df.columns:
            df[col] = df[col].map(quality_map).fillna(0)
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    return df

In [9]:
# preprocess train and test together for consistent encoding
train_c2 = clean_approach2(train.drop(columns=['SalePrice']))
train_c2 = engineer_features(train_c2)
y_train  = np.log(train['SalePrice'])

test_c2 = clean_approach2(test)
test_c2 = engineer_features(test_c2)

combined = pd.concat([train_c2, test_c2], axis=0, ignore_index=True)
combined = encode_features(combined)

X2_train_full = combined.iloc[:len(train_c2)]
X2_test_full  = combined.iloc[len(train_c2):]

# RF feature selection
rf_selector = RandomForestRegressor(n_estimators=100, random_state=42)
rf_selector.fit(X2_train_full, y_train)
importances2 = pd.Series(rf_selector.feature_importances_, index=X2_train_full.columns)
top50_2      = importances2.nlargest(50).index.tolist()

X2_train_top50 = X2_train_full[top50_2]
X2_test_top50  = X2_test_full[top50_2]

# correlation filter on train, apply same columns to test
corr_matrix = X2_train_top50.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
cols_to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]

X2_train_filt = X2_train_top50.drop(columns=cols_to_drop)
X2_test_filt  = X2_test_top50.drop(columns=cols_to_drop)

print(f"Train shape: {X2_train_filt.shape}")
print(f"Test shape:  {X2_test_filt.shape}")

Train shape: (1460, 43)
Test shape:  (1459, 43)


In [23]:
from mlflow.tracking import MlflowClient
import mlflow

MODEL_NAME = "best_house_prices_model"
client = MlflowClient()

model_uri = f"models:/best_house_prices_model/5"
model = mlflow.sklearn.load_model(model_uri)
print("Model loaded successfully")

Model loaded successfully


In [28]:
from sklearn.linear_model import Lasso

best_model_inf = Lasso(alpha=0.001, max_iter=100000)
best_model_inf.fit(X2_train_filt, y_train)

mlflow.set_experiment("lasso_regression")
with mlflow.start_run(run_name="best_model_final_inference"):
    mlflow.sklearn.log_model(
        best_model_inf,
        artifact_path="model",
        registered_model_name="best_house_prices_model"
    )
    print("Registered")

latest = mlflow.tracking.MlflowClient().get_registered_model("best_house_prices_model").latest_versions[-1]
model = mlflow.sklearn.load_model(f"models:/best_house_prices_model/{latest.version}")
print(f"Loaded version {latest.version}")

log_preds = model.predict(X2_test_filt)
preds = np.expm1(log_preds)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
print(submission.head())

2026/04/13 23:45:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 23:45:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'best_house_prices_model' already exists. Creating a new version of this model...
2026/04/13 23:45:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_house_prices_model, version 6
Created version '6' of model 'best_house_prices_model'.


Registered
🏃 View run best_model_final_inference at: https://dagshub.com/lchit22/ml-assignment-house-prices.mlflow/#/experiments/2/runs/8e43d6c536a44810bbe6b0636285d51d
🧪 View experiment at: https://dagshub.com/lchit22/ml-assignment-house-prices.mlflow/#/experiments/2
Loaded version 6
submission.csv saved
     Id      SalePrice
0  1461  115173.379164
1  1462  146495.957765
2  1463  167623.863393
3  1464  195816.836281
4  1465  186399.754662
